# FlowTrack on Kaggle


In [ ]:
!nvidia-smi

In [ ]:
!ffmpeg -version || (apt-get update && apt-get install -y ffmpeg)

In [ ]:
# Clone repo (skip if already mounted in your Colab session)
import os
if not os.path.isdir('/kaggle/working/FlowTrack'):
    !git clone https://github.com/oaboelazm/FlowTrack.git /kaggle/working/FlowTrack
%cd /kaggle/working/FlowTrack

In [ ]:
# Install dependencies
!pip -q install --upgrade pip
!pip -q install -r requirements.txt


In [ ]:
import os
import cv2
import math
import torch
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', 'cuda:0' if torch.cuda.is_available() else 'cpu')


## Training Config
Use smoke mode first to validate the pipeline quickly, then switch to full training.


In [ ]:
from pathlib import Path
import yaml

device = '0' if torch.cuda.is_available() else 'cpu'

detect_cfg = {
    'project': 'runs/flowtrack',
    'name': 'visdrone_D',
    'model': 'yolo26m.pt',
    'data': 'VisDrone.yaml',
    'epochs': 15,
    'imgsz': 640,
    'batch': 16 if device == '0' else 8,
    'device': device,
    'workers': 2,
    'fraction': 0.05,
    'amp': True if device == '0' else False,
    'pretrained': True,
    'save': True,
    'plots': True
}

segment_cfg = {
    'project': 'runs/flowtrack',
    'name': 'coco_S',
    'model': 'yolo26m-seg.pt',
    'data': 'coco.yaml',
    'epochs': 15,
    'imgsz': 640,
    'batch': 16 if device == '0' else 8,
    'device': device,
    'workers': 2,
    'fraction': 0.05,
    'amp': True if device == '0' else False,
    'pretrained': True,
    'save': True,
    'plots': True
}


Path('configs/training').mkdir(parents=True, exist_ok=True)
with open('configs/training/colab_detect.yaml', 'w') as f:
    yaml.safe_dump(detect_cfg, f, sort_keys=False)
with open('configs/training/colab_segment.yaml', 'w') as f:
    yaml.safe_dump(segment_cfg, f, sort_keys=False)    

print('Wrote configs:')
print('- configs/training/colab_detect.yaml')
print('- configs/training/colab_segment.yaml')

# Model Traning

## Detection Model Traning

In [ ]:
# Train detect profile using the config we just wrote
import yaml
cfg = yaml.safe_load(open('configs/training/colab_detect.yaml'))
model_name = cfg.pop('model')
model = YOLO(model_name)
model.train(**cfg)


In [ ]:
# Locate best checkpoint and run validation
from pathlib import Path
import glob

candidates = sorted(glob.glob('/kaggle/working/FlowTrack/runs/detect/runs/flowtrack/visdrone_D/weights/best.pt', recursive=True))
assert candidates, 'No best.pt found. Check training logs.'
best_pt = candidates[-1]
print('Using:', best_pt)

best_model = YOLO(best_pt)
metrics = best_model.val(data='VisDrone.yaml', imgsz=640, device=('0' if torch.cuda.is_available() else 'cpu'))
print(metrics.results_dict)


## Segmentation model training

In [ ]:
# Train segment profile 
import yaml
cfg = yaml.safe_load(open('configs/training/colab_segment.yaml'))
model_name = cfg.pop('model')
model = YOLO(model_name)
model.train(**cfg)


In [ ]:
# Locate best checkpoint and run validation
from pathlib import Path
import glob

candidates = sorted(glob.glob('/kaggle/working/FlowTrack/runs/segment/runs/flowtrack/coco_S/weights/best.pt', recursive=True))
assert candidates, 'No best.pt found. Check training logs.'
best_pt = candidates[-1]
print('Using:', best_pt)

best_model = YOLO(best_pt)
metrics = best_model.val(data='coco.yaml', imgsz=640, device=('0' if torch.cuda.is_available() else 'cpu'))
print(metrics.results_dict)


# Stream Inference (EarthCam)
Important: EarthCam links are tokenized and expire.
If this URL fails, refresh from EarthCam website and paste a fresh link.


In [ ]:
STREAM_URL = 'https://videos-3.earthcam.com/fecnetwork/hdtimes10.flv/chunklist_w511524242.m3u8?t=vBci5OreTDT5OVZWlrH3hFWPpk6y83Y18ohQ4H190JOLnNYuDBmWBdFbyOyxOZMs%2BjXC0vAHjdLTfEjK3qdGvw%3D%3D&td=202603041450'

# EarthCam typically needs headers to avoid 403
os.environ['OPENCV_FFMPEG_CAPTURE_OPTIONS'] = 'user_agent;Mozilla/5.0|referer;https://www.earthcam.com/'

cap = cv2.VideoCapture(STREAM_URL, cv2.CAP_FFMPEG)
print('Stream opened:', cap.isOpened())


# Launch Gradio

The app defaults to chunked + segment playback for smoother display.

Open the generated `gradio.live` link.

In [ ]:
!GRADIO_SHARE=1 python gradio_app.py